In [ ]:
# Cell 1 - Clone repository
%cd /content
!if [ ! -d VDT_GD2 ]; then git clone https://github.com/Darkin72/VDT_GD2.git; fi
%cd /content/VDT_GD2
!git pull


In [ ]:
# Cell 2 - Paper training configuration
DEVICE = "cuda"
WEIGHT_DECAY = 1e-4
MIN_LR = 1e-6
PATIENCE = 50
MULTI_SCALE = "64, 128, 256"
AUGMENTATION = "flip H/V, rotation 0/90/180/270, brightness/contrast +/-0.1"

CONFIG = {
    # Paper: synthetic datasets use 1000 epochs, batch 16 and cosine annealing.
    "SOTS_ITS": {"epochs": 1000, "batch": 16, "lr": 2e-4, "scheduler": "cosine"},
    "SOTS_OTS": {"epochs": 1000, "batch": 16, "lr": 1e-4, "scheduler": "cosine"},
    # I-HAZE/O-HAZY are real-world datasets; use the paper's real-world settings.
    "I_HAZE": {"epochs": 500, "batch": 8, "lr": 2e-4, "scheduler": "step"},
    "O_HAZY": {"epochs": 500, "batch": 8, "lr": 2e-4, "scheduler": "step"},
}
for dataset, cfg in CONFIG.items():
    print(dataset, cfg)
print("device:", DEVICE, "| weight_decay:", WEIGHT_DECAY, "| min_lr:", MIN_LR)
print("multi-scale:", MULTI_SCALE, "| augmentation:", AUGMENTATION)

EVAL_MAX_SIDE = 1024  # Match Bench.ipynb-style bounded full-image evaluation


In [ ]:
# Cell 3 - Train with the paper settings
!pip -q install pillow matplotlib numpy kagglehub tqdm
%cd /content/VDT_GD2
import os
from pathlib import Path
import kagglehub

its_root = Path(kagglehub.dataset_download("balraj98/indoor-training-set-its-residestandard"))
ots_root = Path(kagglehub.dataset_download("brunobelloni/outdoor-training-set-ots-reside"))
print("ITS path:", its_root)
print("OTS path:", ots_root)
os.environ["HW_ITS_ROOT"] = str(its_root)
os.environ["HW_OTS_ROOT"] = str(ots_root)
os.environ["HW_DEVICE"] = DEVICE
os.environ["HW_WD"] = str(WEIGHT_DECAY)
os.environ["HW_MIN_LR"] = str(MIN_LR)
os.environ["HW_PATIENCE"] = str(PATIENCE)

c = CONFIG["I_HAZE"]
os.environ["HW_EPOCHS"] = str(c["epochs"]); os.environ["HW_BATCH"] = str(c["batch"]); os.environ["HW_LR"] = str(c["lr"]); os.environ["HW_SCHEDULER"] = c["scheduler"]
!python -m solution.wavelet_dehaze.train dataset/I-HAZE/train --val-data dataset/I-HAZE/val --device "$HW_DEVICE" --epochs "$HW_EPOCHS" --batch-size "$HW_BATCH" --size 256 --lr "$HW_LR" --weight-decay "$HW_WD" --scheduler "$HW_SCHEDULER" --step-size 100 --min-lr "$HW_MIN_LR" --augment --multi-scale --patience "$HW_PATIENCE" --out haze_wavelet_i_haze.pt --history history_i_haze.json

c = CONFIG["O_HAZY"]
os.environ["HW_EPOCHS"] = str(c["epochs"]); os.environ["HW_BATCH"] = str(c["batch"]); os.environ["HW_LR"] = str(c["lr"]); os.environ["HW_SCHEDULER"] = c["scheduler"]
!python -m solution.wavelet_dehaze.train dataset/O-HAZY/train --val-data dataset/O-HAZY/val --device "$HW_DEVICE" --epochs "$HW_EPOCHS" --batch-size "$HW_BATCH" --size 256 --lr "$HW_LR" --weight-decay "$HW_WD" --scheduler "$HW_SCHEDULER" --step-size 100 --min-lr "$HW_MIN_LR" --augment --multi-scale --patience "$HW_PATIENCE" --out haze_wavelet_o_hazy.pt --history history_o_hazy.json


c = CONFIG["SOTS_ITS"]
os.environ["HW_EPOCHS"] = str(c["epochs"]); os.environ["HW_BATCH"] = str(c["batch"]); os.environ["HW_LR"] = str(c["lr"]); os.environ["HW_SCHEDULER"] = c["scheduler"]
!python -m solution.wavelet_dehaze.train "$HW_ITS_ROOT" --device "$HW_DEVICE" --epochs "$HW_EPOCHS" --batch-size "$HW_BATCH" --size 256 --lr "$HW_LR" --weight-decay "$HW_WD" --scheduler "$HW_SCHEDULER" --min-lr "$HW_MIN_LR" --augment --multi-scale --patience "$HW_PATIENCE" --out haze_wavelet_sots_its.pt --history history_sots_its.json

c = CONFIG["SOTS_OTS"]
os.environ["HW_EPOCHS"] = str(c["epochs"]); os.environ["HW_BATCH"] = str(c["batch"]); os.environ["HW_LR"] = str(c["lr"]); os.environ["HW_SCHEDULER"] = c["scheduler"]
!python -m solution.wavelet_dehaze.train "$HW_OTS_ROOT" --device "$HW_DEVICE" --epochs "$HW_EPOCHS" --batch-size "$HW_BATCH" --size 256 --lr "$HW_LR" --weight-decay "$HW_WD" --scheduler "$HW_SCHEDULER" --min-lr "$HW_MIN_LR" --augment --multi-scale --patience "$HW_PATIENCE" --out haze_wavelet_sots_ots.pt --history history_sots_ots.json


In [ ]:
# Cell 4 - Preview metrics separately for each dataset
import json
from pathlib import Path
import matplotlib.pyplot as plt

files = {
    "I-HAZE": "history_i_haze.json",
    "O-HAZY": "history_o_hazy.json",
    "SOTS-ITS": "history_sots_its.json",
    "SOTS-OTS": "history_sots_ots.json",
}
histories = {
    name: json.loads(Path(path).read_text())
    for name, path in files.items()
    if Path(path).exists()
}

if not histories:
    raise FileNotFoundError("No history_*.json files found. Run Cell 3 first.")

fig, axes = plt.subplots(
    len(histories), 3,
    figsize=(20, 5 * len(histories)),
    squeeze=False,
)
for row, (name, records) in enumerate(histories.items()):
    epoch = [item["epoch"] for item in records]
    metrics = [
        ("train_loss", "val_loss", "Loss"),
        ("train_psnr", "val_psnr", "PSNR (dB)"),
        ("train_ssim", "val_ssim", "SSIM"),
    ]
    for col, (train_key, val_key, title) in enumerate(metrics):
        ax = axes[row, col]
        ax.plot(epoch, [item[train_key] for item in records], label="Train", linewidth=2)
        ax.plot(epoch, [item[val_key] for item in records], "--", label="Validation", linewidth=2)
        ax.set_title(f"{name} - {title}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(title)
        ax.grid(alpha=0.25)
        ax.legend()

plt.tight_layout()
plt.savefig("haze_wavelet_training_metrics_by_dataset.png", dpi=160)
plt.show()

In [ ]:
# Cell 5 - Benchmark: 4 Excel reports (dataset, fog, domain gap, hardware)
!pip -q install kagglehub
import kagglehub
sots_download = Path(kagglehub.dataset_download("balraj98/synthetic-objective-testing-set-sots-reside"))
print("SOTS path:", sots_download)
sots_root = Path("dataset/Synthetic Objective Testing Set (SOTS) [RESIDE]")
for domain in ("indoor", "outdoor"):
    for kind in ("clear", "hazy"):
        target = sots_root / domain / "test" / kind
        target.parent.mkdir(parents=True, exist_ok=True)
        source = sots_download / domain / kind
        if not source.is_dir(): raise FileNotFoundError(source)
        if not target.exists(): target.symlink_to(source, target_is_directory=True)

from pathlib import Path
!pip -q install openpyxl psutil scikit-image thop
%cd /content/VDT_GD2

RESULT_DIR = "utils/evaluation_results/hazewavenet"
import os
os.environ["HW_DEVICE"] = DEVICE
os.environ["HW_MAX_SIDE"] = str(EVAL_MAX_SIDE)
os.environ["HW_RESULT_DIR"] = RESULT_DIR
!mkdir -p "$HW_RESULT_DIR"

!python -m solution.wavelet_dehaze.evaluate   --data-root dataset   --output-dir "$HW_RESULT_DIR"   --checkpoint-i-haze haze_wavelet_i_haze.pt   --checkpoint-o-hazy haze_wavelet_o_hazy.pt   --checkpoint-sots-its haze_wavelet_sots_its.pt   --checkpoint-sots-ots haze_wavelet_sots_ots.pt   --device "$HW_DEVICE"   --max-side "$HW_MAX_SIDE"

# Add model complexity to the hardware report.
import pandas as pd
import torch
from thop import profile as thop_profile
from solution.wavelet_dehaze.model import HazeWaveNet

hardware_rows = []
for input_side in (256, 1024):
    model = HazeWaveNet().to(DEVICE).eval()
    checkpoint = Path("haze_wavelet_i_haze.pt")
    state = torch.load(checkpoint, map_location=DEVICE)
    model.load_state_dict(state.get("model", state))
    params = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    dummy = torch.zeros(1, 3, input_side, input_side, device=DEVICE)
    with torch.inference_mode():
        macs, _ = thop_profile(model, inputs=(dummy,), verbose=False)
    hardware_rows.append({
        "model": "HazeWaveNet", "checkpoint": checkpoint.name,
        "input_shape": f"1x3x{input_side}x{input_side}",
        "parameters": params, "trainable_parameters": trainable,
        "gmacs": macs / 1e9, "gflops": 2 * macs / 1e9,
        "device": DEVICE,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    })
hardware_df = pd.DataFrame(hardware_rows)
hardware_path = Path(RESULT_DIR) / "hardware.xlsx"
hardware_df.to_csv(Path(RESULT_DIR) / "hardware.csv", index=False, encoding="utf-8-sig")
with pd.ExcelWriter(hardware_path, engine="openpyxl", mode="a" if hardware_path.exists() else "w", if_sheet_exists="replace" if hardware_path.exists() else None) as writer:
    hardware_df.to_excel(writer, sheet_name="Hardware", index=False)
print("Hardware report:")
print(hardware_df.to_string(index=False))

print("Excel reports:")
for path in sorted(Path(RESULT_DIR).glob("*.xlsx")):
    print(" -", path)


In [ ]:
# Cell 6 - Detailed stage timing (preprocess + model decomposition/branches)
import time
import numpy as np
import torch
from PIL import Image
from solution.wavelet_dehaze.model import HazeWaveNet

TIMING_REPEATS = 10
WARMUP = 3
TIMING_SIZE = 256  # Match training input; set to 1024 to profile bounded evaluation.

def timed_preprocess(path):
    start = time.perf_counter()
    with Image.open(path) as image:
        image = image.convert("RGB").resize((TIMING_SIZE, TIMING_SIZE), Image.Resampling.BILINEAR)
    tensor = torch.from_numpy(np.array(image, copy=True)).float().div(255).permute(2, 0, 1).unsqueeze(0)
    return tensor, (time.perf_counter() - start) * 1000

def profile_dataset(label, checkpoint, folder):
    model = HazeWaveNet().to(DEVICE).eval()
    state = torch.load(checkpoint, map_location=DEVICE)
    model.load_state_dict(state.get("model", state))
    paths = sorted(p for p in Path(folder).iterdir() if p.is_file())
    if not paths:
        raise FileNotFoundError(folder)
    sample, _ = timed_preprocess(paths[0]); sample = sample.to(DEVICE)
    for _ in range(WARMUP): model(sample)
    rows = []
    for path in paths:
        image, preprocess_ms = timed_preprocess(path)
        image = image.to(DEVICE)
        _, stage = model.profile_forward(image, repeats=TIMING_REPEATS)
        stage["preprocess"] = preprocess_ms
        stage["total_with_preprocess"] = preprocess_ms + stage["total"]
        rows.append(stage)
    keys = ["preprocess", "dwt_mfde_decomposition", "dcp_fixed_prior", "low_frequency_fegg_fegb", "high_frequency_vp_vr", "wim_reconstruction", "final_refinement", "total", "total_with_preprocess"]
    print(f"\n{label} ({len(rows)} images, size={TIMING_SIZE}x{TIMING_SIZE}, device={DEVICE})")
    for key in keys:
        print(f"{key:32s}: {np.mean([row[key] for row in rows]):9.3f} ms")

profile_dataset("I-HAZE", "haze_wavelet_i_haze.pt", "dataset/I-HAZE/test/hazy")
profile_dataset("O-HAZY", "haze_wavelet_o_hazy.pt", "dataset/O-HAZY/test/hazy")
